# 01 — cluster-cluster correlations (alignment study)

**Feeds:** Fig 4e (equivalent output)

**Position in the chain:** run the numbered stages in order

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

This notebook ran on the earlier human embryo clustering object and the `07b` and `07c` labels built from it (`parity/results/intermediates/`), so it does not run on the embryo object `07_human_embryo.ipynb` builds here. The earlier object is in the Zenodo deposit (`../../data/DOWNLOAD.md`); `scrnaseq/run_chain.py` runs this notebook, and its `07b`/`07c` dependencies, when it is present under `SCRNASEQ_INPUT_ROOT`, and skips them otherwise. Fig 4d and 4e are drawn from its saved tables in `scrnaseq/morph_embryo_correspondence/derived/` when this notebook does not run.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original searched upward for the analysis directory (the one holding `PROVENANCE_MAP.md`, `parity/` and `trunk_main_dev/`). `ANALYSIS_ROOT` stands in for that directory: it is `$SCRNASEQ_RESULTS_ROOT` (default `scrnaseq/output/`), where the chain's notebooks write in the same layout, and this notebook's outputs go under its `experiments/rq1_5_human_embryo_alignment/`. The helper modules are imported from `scrnaseq/morph_embryo_correspondence/scripts/` instead of `experiments/rq1_5_human_embryo_alignment/src/`.

2. The embryo clusters object and its SMD companion. `SCRNASEQ_INPUT_ROOT` (default `data/scrnaseq_inputs/`) is checked for `earlier_embryo_run/results/intermediates/07_human_embryo/{adata_embryo_with_clusters,adata_embryo_SMD}.h5ad`; when absent, the notebook falls back to `embryo_path`, as before this override existed (in practice unreachable, since `run_chain.py` only runs this notebook when the object is present).

No other line of code was changed.


# 01 - Cluster-Cluster Correlations

This notebook is the cluster-level gene-set sweep for the human embryo alignment experiment.

It now shares the same effective cluster definitions as notebook `12`:
- trunk morph clusters from `trunk_main_dev/02_trunk_main`
- human embryo clusters from `parity/07_human_embryo`
- local embryo refinements from `07b` (Floor Plate) and `07c` (Posterior NT / NMP)

Compared with notebook `12`, this stage is broader rather than more final:
1. compute cluster means on the final morph/embryo cluster definitions
2. vary the gene set used for the cluster-cluster correlation
3. export top-match and reciprocal-match summaries across gene sets
4. keep cluster-level reciprocal NNLS as a secondary analysis arm

This notebook intentionally does **not** include the older cell-level mapping and projection sections; those belong in notebook `02`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src' / 'trunk_morph_ref').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'src' / 'trunk_morph_ref').exists(), f'Could not locate repository root from {Path.cwd()}'
sys.path.insert(0, str(REPO_ROOT))
from src.trunk_morph_ref.paths import scrnaseq_input_root, scrnaseq_results_root

# Stands in for the original analysis directory: the chain's outputs, in the same layout.
ANALYSIS_ROOT = scrnaseq_results_root(REPO_ROOT)
# The earlier human embryo clusters object, fetched from the Zenodo deposit (see data/DOWNLOAD.md).
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO_ROOT)

EXPERIMENT_ROOT = ANALYSIS_ROOT / 'experiments' / 'rq1_5_human_embryo_alignment'
STAGE_NAME = '01_cluster_cluster_correlations'
STAGE_DIR = EXPERIMENT_ROOT / 'results' / 'intermediates' / STAGE_NAME
FIG_DIR = STAGE_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'scrnaseq' / 'morph_embryo_correspondence' / 'scripts'))

from manuscript_cluster_cluster_correlation import (
    MORPH_DISPLAY_LABELS,
    apply_morph_display_order,
    build_gene_sets,
    cluster_averages,
    cross_cluster_correlation,
    plot_correlation_heatmap,
    plot_gene_set_grid,
)
from cluster_cluster_correlation import (
    summarize_top_matches,
    summarize_column_best_matches,
    summarize_reciprocal_best_matches,
    invert_expected_mapping,
    nnls_cluster_regression,
    summarize_reciprocal_nnls,
)
from merged_embryo_cluster_cluster_correlation import (
    EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
    MERGED_EMBRYO_DISPLAY_LABELS_V3,
    MERGED_EMBRYO_ORDER_V3,
    assign_merged_embryo_labels_v3,
    load_floor_plate_reassignment_07b,
    load_nmp_reassignment_07c,
    merged_cluster_averages_v3,
    merged_embryo_membership_table_v3,
    merged_group_counts_v3,
    merged_group_summary_table_v3,
    validate_merge_scheme_v3,
)
from src.trunk_morph_ref.pipeline_io import load_h5ad, load_pickle, save_json


## Canonical Inputs

In [ ]:
trunk_path = ANALYSIS_ROOT / 'trunk_main_dev' / 'results' / 'intermediates' / '02_trunk_main'
embryo_path = ANALYSIS_ROOT / 'parity' / 'results' / 'intermediates' / '07_human_embryo'
floor_plate_stage_path = ANALYSIS_ROOT / 'parity' / 'results' / 'intermediates' / '07b_human_embryo_floor_plate_subclustering'
floor_plate_reassign_path = floor_plate_stage_path / 'tables' / 'ivsc_cluster_labels.csv'
nmp_stage_path = ANALYSIS_ROOT / 'parity' / 'results' / 'intermediates' / '07c_human_embryo_nmp_subclustering'
nmp_reassign_path = nmp_stage_path / 'tables' / 'nmp_cluster_labels.csv'

adata_morph_with_clusters = load_h5ad(trunk_path / 'adata_morph_with_clusters.h5ad')
adata_morph_smd = load_h5ad(trunk_path / 'adata_morph_SMD.h5ad')

# This stage ran on the earlier human embryo clustering object; nothing in this chain writes it
# to embryo_path (the parity lineage is not rebuilt here). Read it, and its SMD companion, from
# the Zenodo deposit when present, otherwise fall back to embryo_path, as before this override
# existed.
_earlier_embryo_run_path = (
    SCRNASEQ_INPUT_ROOT / 'earlier_embryo_run' / 'results' / 'intermediates' / '07_human_embryo'
)
if (_earlier_embryo_run_path / 'adata_embryo_with_clusters.h5ad').exists():
    adata_embryo_with_clusters = load_h5ad(_earlier_embryo_run_path / 'adata_embryo_with_clusters.h5ad')
    adata_embryo_smd = load_h5ad(_earlier_embryo_run_path / 'adata_embryo_SMD.h5ad')
else:
    adata_embryo_with_clusters = load_h5ad(embryo_path / 'adata_embryo_with_clusters.h5ad')
    adata_embryo_smd = load_h5ad(embryo_path / 'adata_embryo_SMD.h5ad')

integration_gene_panel = load_pickle(trunk_path / 'integration_gene_panel.pkl')
morph_order = apply_morph_display_order(list(load_pickle(trunk_path / 'celltypeorder.pkl')))
embryo_order_all = list(adata_embryo_with_clusters.obs['leiden_embryo'].cat.categories)

floor_plate_reassignment_07b = load_floor_plate_reassignment_07b(floor_plate_reassign_path)
nmp_reassignment_07c = load_nmp_reassignment_07c(nmp_reassign_path)

summary = pd.DataFrame({
    'dataset': ['morph_full', 'morph_smd', 'embryo_full', 'embryo_smd', '07b_ivsc_assignments', '07c_pnt_nmp_assignments'],
    'n_cells': [
        adata_morph_with_clusters.n_obs,
        adata_morph_smd.n_obs,
        adata_embryo_with_clusters.n_obs,
        adata_embryo_smd.n_obs,
        int(floor_plate_reassignment_07b.shape[0]),
        int(nmp_reassignment_07c.shape[0]),
    ],
    'n_genes': [
        adata_morph_with_clusters.n_vars,
        adata_morph_smd.n_vars,
        adata_embryo_with_clusters.n_vars,
        adata_embryo_smd.n_vars,
        pd.NA,
        pd.NA,
    ],
})
display(summary)


## Final Embryo Group Definitions

In [ ]:
missing_labels, unused_labels = validate_merge_scheme_v3(embryo_order_all)
assert not missing_labels, f'Merge scheme is missing embryo labels: {missing_labels}'

adata_embryo_with_clusters.obs['merged_embryo_v3'] = assign_merged_embryo_labels_v3(
    adata_embryo_with_clusters.obs[['leiden_embryo']],
    floor_plate_reassignment_07b,
    nmp_reassignment_07c,
).values

group_counts = merged_group_counts_v3(adata_embryo_with_clusters)
group_totals = (
    group_counts.groupby('merged_embryo_v3', observed=True)['n_cells']
    .sum()
    .rename('n_cells_total')
    .reset_index()
    .rename(columns={'merged_embryo_v3': 'merged_group'})
)
membership = merged_embryo_membership_table_v3().sort_values(['status', 'merged_group', 'assignment_rule', 'embryo_label']).reset_index(drop=True)
group_summary = merged_group_summary_table_v3()

membership.to_csv(STAGE_DIR / 'merged_embryo_membership_table.csv', index=False)
group_summary.to_csv(STAGE_DIR / 'merged_embryo_group_summary.csv', index=False)
group_counts.to_csv(STAGE_DIR / 'merged_embryo_group_counts_by_source_cluster.csv', index=False)
group_totals.to_csv(STAGE_DIR / 'merged_embryo_group_cell_totals.csv', index=False)
save_json({
    'missing_labels': missing_labels,
    'unused_labels': unused_labels,
    'merged_embryo_order': MERGED_EMBRYO_ORDER_V3,
    'floor_plate_override_source': str(floor_plate_reassign_path),
    'nmp_override_source': str(nmp_reassign_path),
}, STAGE_DIR / 'merge_validation.json')

display(group_totals)
display(group_summary)


## Fixed Mapping Dictionary

In [ ]:
expected_mapping_df = pd.DataFrame(
    [
        {'morph_label': morph_label, 'expected_embryo_labels': ' | '.join(embryo_labels)}
        for morph_label, embryo_labels in EXPECTED_MERGED_EMBRYO_BY_MORPH_V3.items()
    ]
)
display(expected_mapping_df)
expected_mapping_df.to_csv(STAGE_DIR / 'expected_mapping_dictionary.csv', index=False)


## Gene-Set Variants

In [ ]:
gene_sets = build_gene_sets(
    adata_morph_full=adata_morph_with_clusters,
    adata_embryo_full=adata_embryo_with_clusters,
    adata_morph_smd=adata_morph_smd,
    adata_embryo_smd=adata_embryo_smd,
    integration_gene_panel=integration_gene_panel,
)

gene_set_summary = (
    pd.DataFrame([
        {'gene_set': name, 'n_genes': len(genes)}
        for name, genes in gene_sets.items()
    ])
    .sort_values('n_genes')
    .reset_index(drop=True)
)
display(gene_set_summary)
gene_set_summary.to_csv(STAGE_DIR / 'gene_set_summary.csv', index=False)


## Cluster Averages

In [ ]:
avg_morph = cluster_averages(adata_morph_with_clusters, cluster_key='leiden_morph').loc[morph_order]
avg_embryo = merged_cluster_averages_v3(
    adata_embryo_with_clusters,
    floor_plate_reassignment_07b,
    nmp_reassignment_07c,
    merged_order=MERGED_EMBRYO_ORDER_V3,
)

avg_morph.to_csv(STAGE_DIR / 'avg_morph_final_clusters.csv')
avg_embryo.to_csv(STAGE_DIR / 'avg_embryo_final_clusters.csv')

print(avg_morph.shape, avg_embryo.shape)
display(avg_morph.head())
display(avg_embryo.head())


## Correlation Matrices Across Gene Sets

In [ ]:
corr_results = {}
overview_rows = []

for gene_set_name, genes in gene_sets.items():
    corr_df = cross_cluster_correlation(avg_morph, avg_embryo, genes)
    corr_results[gene_set_name] = corr_df
    corr_df.to_csv(STAGE_DIR / f'corr_final_embryo__{gene_set_name}.csv')

    row_top = summarize_top_matches(corr_df, expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3)
    col_top = summarize_column_best_matches(corr_df)
    reciprocal, reciprocal_pairs = summarize_reciprocal_best_matches(
        corr_df,
        expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
    )

    row_top.to_csv(STAGE_DIR / f'row_top_matches_final_embryo__{gene_set_name}.csv', index=False)
    col_top.to_csv(STAGE_DIR / f'col_top_matches_final_embryo__{gene_set_name}.csv', index=False)
    reciprocal.to_csv(STAGE_DIR / f'reciprocal_top_matches_final_embryo__{gene_set_name}.csv', index=False)
    reciprocal_pairs.to_csv(STAGE_DIR / f'reciprocal_pairs_final_embryo__{gene_set_name}.csv', index=False)

    expected_top1_rate = row_top['expected_contains_top1'].dropna().mean()
    reciprocal_top1_rate = reciprocal['reciprocal_top1'].mean()
    expected_reciprocal_rate = reciprocal['expected_reciprocal_top1'].dropna().mean()
    overview_rows.append(
        {
            'gene_set': gene_set_name,
            'n_genes': len(genes),
            'expected_top1_rate_final_embryo': None if pd.isna(expected_top1_rate) else float(expected_top1_rate),
            'reciprocal_top1_rate_final_embryo': float(reciprocal_top1_rate),
            'expected_reciprocal_top1_rate_final_embryo': None if pd.isna(expected_reciprocal_rate) else float(expected_reciprocal_rate),
            'median_expected_best_corr': float(row_top['expected_best_corr'].dropna().median()),
            'median_top1_minus_top2': float(row_top['top1_minus_top2'].dropna().median()),
        }
    )

overview_df = pd.DataFrame(overview_rows).sort_values('n_genes').reset_index(drop=True)
display(overview_df)
overview_df.to_csv(STAGE_DIR / 'correlation_overview.csv', index=False)

display(corr_results['integration_gene_panel'])
display(pd.read_csv(STAGE_DIR / 'row_top_matches_final_embryo__integration_gene_panel.csv'))


## Reciprocal Best-Match Summary

In [ ]:
plot_df = overview_df.copy()
label_map = {
    'integration_gene_panel': 'Integration panel',
    'smd_intersection': 'SMD intersection',
    'smd_union_shared': 'SMD union',
    'all_shared_genes': 'All shared genes',
}
plot_df['display_label'] = plot_df['gene_set'].map(label_map).fillna(plot_df['gene_set'])
plot_df = plot_df.sort_values('n_genes').reset_index(drop=True)
plot_df['top1_pct'] = 100 * plot_df['expected_top1_rate_final_embryo']
plot_df['reciprocal_pct'] = 100 * plot_df['expected_reciprocal_top1_rate_final_embryo']

y = range(len(plot_df))
fig, ax = plt.subplots(figsize=(8.8, 3.6 + 0.45 * len(plot_df)), dpi=300)
ax.barh(y, plot_df['top1_pct'], height=0.55, color='#d9d9d9', edgecolor='black', label='Expected top-1')
ax.barh(y, plot_df['reciprocal_pct'], height=0.32, color='#4C78A8', edgecolor='black', label='Expected reciprocal top-1')
for yy, top1, recip, n_genes in zip(y, plot_df['top1_pct'], plot_df['reciprocal_pct'], plot_df['n_genes']):
    ax.text(top1 + 1.0, yy + 0.12, f'{top1:.0f}%', va='center', fontsize=9)
    ax.text(recip + 1.0, yy - 0.12, f'{recip:.0f}%', va='center', fontsize=9)
    ax.text(101.5, yy, f'n={n_genes}', va='center', ha='left', fontsize=8, color='#555555')
ax.set_yticks(list(y))
ax.set_yticklabels(plot_df['display_label'])
ax.invert_yaxis()
ax.set_xlim(0, 108)
ax.set_xlabel('Rate (%)')
ax.set_title('Reciprocal best-match summary | final embryo groups')
ax.legend(loc='lower right', frameon=False)
ax.grid(axis='x', linestyle=':', alpha=0.4)
fig.tight_layout()
fig.savefig(FIG_DIR / 'reciprocal_rate_summary_final.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
fig.savefig(FIG_DIR / 'reciprocal_rate_summary_final.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()
plt.close(fig)


## Cluster-Level Reciprocal NNLS

In [ ]:
expected_morph_by_embryo = invert_expected_mapping(EXPECTED_MERGED_EMBRYO_BY_MORPH_V3)

nnls_overview_rows = []
for gene_set_name, genes in gene_sets.items():
    morph_weights, morph_summary = nnls_cluster_regression(
        avg_morph,
        avg_embryo,
        genes,
        expected_by_target=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
        target_label_name='morph',
        source_label_name='embryo',
        standardize=True,
    )
    embryo_weights, embryo_summary = nnls_cluster_regression(
        avg_embryo,
        avg_morph,
        genes,
        expected_by_target=expected_morph_by_embryo,
        target_label_name='embryo',
        source_label_name='morph',
        standardize=True,
    )
    reciprocal_summary, reciprocal_pairs = summarize_reciprocal_nnls(morph_summary, embryo_summary)

    morph_weights.to_csv(STAGE_DIR / f'nnls_weights_morph_from_embryo__{gene_set_name}.csv')
    embryo_weights.to_csv(STAGE_DIR / f'nnls_weights_embryo_from_morph__{gene_set_name}.csv')
    morph_summary.to_csv(STAGE_DIR / f'nnls_summary_morph_from_embryo__{gene_set_name}.csv', index=False)
    embryo_summary.to_csv(STAGE_DIR / f'nnls_summary_embryo_from_morph__{gene_set_name}.csv', index=False)
    reciprocal_summary.to_csv(STAGE_DIR / f'nnls_reciprocal_top_matches_final_embryo__{gene_set_name}.csv', index=False)
    reciprocal_pairs.to_csv(STAGE_DIR / f'nnls_reciprocal_pairs_final_embryo__{gene_set_name}.csv', index=False)

    expected_top1_rate = morph_summary['expected_contains_top1'].dropna().mean()
    reciprocal_top1_rate = reciprocal_summary['reciprocal_top1'].mean()
    expected_reciprocal_rate = reciprocal_summary['expected_reciprocal_top1'].dropna().mean()
    nnls_overview_rows.append(
        {
            'gene_set': gene_set_name,
            'n_genes': len(genes),
            'expected_top1_rate_final_embryo': None if pd.isna(expected_top1_rate) else float(expected_top1_rate),
            'reciprocal_top1_rate_final_embryo': float(reciprocal_top1_rate),
            'expected_reciprocal_top1_rate_final_embryo': None if pd.isna(expected_reciprocal_rate) else float(expected_reciprocal_rate),
            'median_expected_weight_sum_morph_from_embryo': float(morph_summary['expected_weight_sum'].dropna().median()),
            'median_fit_corr_morph_from_embryo': float(morph_summary['fit_corr'].dropna().median()),
            'median_fit_r2_morph_from_embryo': float(morph_summary['fit_r2'].dropna().median()),
        }
    )

nnls_overview_df = pd.DataFrame(nnls_overview_rows).sort_values('n_genes').reset_index(drop=True)
display(nnls_overview_df)
nnls_overview_df.to_csv(STAGE_DIR / 'nnls_overview_final_embryo.csv', index=False)

display(pd.read_csv(STAGE_DIR / 'nnls_reciprocal_top_matches_final_embryo__integration_gene_panel.csv'))


## Figure Exports

In [ ]:
selected_gene_sets = [
    'integration_gene_panel',
    'smd_intersection',
    'smd_union_shared',
    'all_shared_genes',
]
selected_corr_results = {name: corr_results[name] for name in selected_gene_sets}

fig = plot_gene_set_grid(
    selected_corr_results,
    title='Trunk morph vs human embryo cluster correlations across gene sets',
    expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
    morph_label_lookup=MORPH_DISPLAY_LABELS,
    embryo_label_lookup=MERGED_EMBRYO_DISPLAY_LABELS_V3,
    x_label='Trunk morph cluster',
    y_label='Human embryo cluster',
    output_path=FIG_DIR / 'grid_final_gene_sets.png',
)
fig.savefig(FIG_DIR / 'grid_final_gene_sets.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()
plt.close(fig)

for gene_set_name, corr_df in corr_results.items():
    fig = plot_correlation_heatmap(
        corr_df,
        title=f'Trunk morph vs human embryo cluster correlations | {gene_set_name} (n={len(gene_sets[gene_set_name])} genes)',
        expected_by_morph=EXPECTED_MERGED_EMBRYO_BY_MORPH_V3,
        morph_label_lookup=MORPH_DISPLAY_LABELS,
        embryo_label_lookup=MERGED_EMBRYO_DISPLAY_LABELS_V3,
        x_label='Trunk morph cluster',
        y_label='Human embryo cluster',
        output_path=FIG_DIR / f'final_embryo__{gene_set_name}.png',
    )
    fig.savefig(FIG_DIR / f'final_embryo__{gene_set_name}.pdf', bbox_inches='tight', pad_inches=0.02)
    plt.close(fig)

print(FIG_DIR)
print(sorted(p.name for p in FIG_DIR.glob('*.png')))


## Save Notebook Metadata

In [ ]:
save_json(
    {
        'stage': STAGE_NAME,
        'description': 'Cluster-level gene-set sweep using the same final morph and embryo cluster definitions as notebook 12.',
        'morph_input': str(trunk_path / 'adata_morph_with_clusters.h5ad'),
        'morph_smd_input': str(trunk_path / 'adata_morph_SMD.h5ad'),
        'embryo_input': str(embryo_path / 'adata_embryo_with_clusters.h5ad'),
        'embryo_smd_input': str(embryo_path / 'adata_embryo_SMD.h5ad'),
        'floor_plate_override_input': str(floor_plate_reassign_path),
        'nmp_override_input': str(nmp_reassign_path),
        'morph_order': morph_order,
        'merged_embryo_order': MERGED_EMBRYO_ORDER_V3,
        'gene_sets': {name: len(genes) for name, genes in gene_sets.items()},
        'selected_gene_sets': selected_gene_sets,
    },
    STAGE_DIR / 'meta.json',
)

print(STAGE_DIR)
